# DC-Net (Li et al., 2023)

A 5-level U-Net with batch normalisation trained with an L2 loss on the 22,000 training patches to map 800 m upward-continued
data to the ground-level field. Weights: `weights/dcnet.pth`.

**Evaluation protocol (all notebooks).** The 10 field test patches (32 × 32 nodes, 400 m spacing) are
scaled to [-1, 1] per patch. PSNR, SSIM, relative error (RE) and RMSE are computed on the central 24 × 24
window and reported as mean ± standard deviation (ddof = 0) over the 10 patches. Networks run in full FP32
with deterministic cuDNN (set when `gravdc` is imported, see `gravdc/precision.py`).

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))  # make the gravdc package importable

import numpy as np
import torch

from gravdc import evaluate, format_summary, load_model, load_test_set, save_result, summarize
from gravdc.plotting import show_reconstruction
from gravdc.table3 import compare

DATA_DIR = "../data/test"
RESULTS_DIR = "../results"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

device: cuda | torch 2.9.0+cu128


## Data
Observation: the ground-level field continued 800 m upward in a single step (zero padding to 36 × 36).

In [2]:
from gravdc import upward_continue

names, x = load_test_set(DATA_DIR)
x = x.to(device)
y = upward_continue(x, 800)
print(len(names), "patches:", names)

10 patches: ['data14.mat', 'data16.mat', 'data18.mat', 'data20.mat', 'data22.mat', 'data24.mat', 'data26.mat', 'data28.mat', 'data30.mat', 'data32.mat']


## Model and inference

In [3]:
model = load_model("dcnet", device)
with torch.no_grad():
    x_hat = model(y)

## Metrics

In [4]:
per_image = evaluate(x, x_hat)
compare("DC-Net", summarize(per_image))
save_result("DC-Net", per_image, names, RESULTS_DIR)

DC-Net
  metric             this run              Table 3
  PSNR       33.9575 ± 1.9905     33.9575 ± 1.9905  ✓
  SSIM        0.9771 ± 0.0116      0.9771 ± 0.0116  ✓
  RE          0.1003 ± 0.0208      0.1003 ± 0.0208  ✓
  RMSE        0.0353 ± 0.0101      0.0353 ± 0.0101  ✓


In [5]:
# The field data cannot be redistributed, so figures of them are off by default (executed notebooks can then
# be shared without them). Set SHOW_FIGURE = True to look at a reconstruction.
SHOW_FIGURE = False
if SHOW_FIGURE:
    k = 0
    show_reconstruction(x[k], y[k], {"DC-Net": x_hat[k]}, title=names[k])